# Step-by-Step Scaler Development
This notebook is used to interactively build and validate the from-scratch `StandardScaler` and Label Encoder logic. This ensures full transparency of the underlying mathematics (Z-score standardization) before exporting to `src/scaler.py`.

## Step 2: Label Encoding

Neural networks, including MLPs, are mathematical models that operate strictly on numerical data. During the forward pass, inputs are multiplied by numerical weights ($z = \sum (x_k \cdot w_k) + b$). String labels like 'M' (Malignant) and 'B' (Benign) cannot be processed in this way.

Furthermore, for binary classification using Binary Cross-Entropy loss ($E = -\frac{1}{N} \sum [y_n \log(p_n) + (1 - y_n) \log(1 - p_n)]$), the true labels ($y_n$) must be binary values ($0$ or $1$) so that the mathematical formulation of logarithmic penalty works correctly.

We will map:
- 'M' (Malignant) $\rightarrow 1$ (Positive class)
- 'B' (Benign) $\rightarrow 0$ (Negative class)

In [1]:
import pandas as pd

In [12]:
import numpy as np


In [5]:
train_df = pd.read_csv('../data/training_data.csv')

In [7]:
train_df.head()

,88995002,M,20.73,31.12,135.7,1419.0,0.09469,0.1143,0.1367,0.08646,...,32.49,47.16,214.0,3432.0,0.1401,0.2644,0.3442,0.1659,0.2868,0.08218
0,859471,B,9.029,17.33,58.79,250.5,0.10660,0.14130,0.31300,0.04375,...,10.31,22.65,65.50,324.7,0.14820,0.43650,1.25200,0.17500,0.4228,0.11750
1,873593,M,21.090,26.57,142.70,1311.0,0.11410,0.28320,0.24870,0.14960,...,26.68,33.48,176.50,2089.0,0.14910,0.75840,0.67800,0.29030,0.4098,0.12840
2,859196,B,9.173,13.86,59.20,260.9,0.07721,0.08751,0.05988,0.02180,...,10.01,19.23,65.59,310.1,0.09836,0.16780,0.13970,0.05087,0.3282,0.08490
3,88466802,B,10.650,25.22,68.01,347.0,0.09657,0.07234,0.02379,0.01615,...,12.25,35.19,77.98,455.7,0.14990,0.13980,0.11250,0.06136,0.3409,0.08147
4,858970,B,10.170,14.88,64.55,311.9,0.11340,0.08061,0.01084,0.01290,...,11.02,17.45,69.86,368.6,0.12750,0.09866,0.02168,0.02579,0.3557,0.08020


In [8]:
train_labels = train_df.iloc[:, 1].values

In [10]:
train_labels[:5]

<StringArray>
['B', 'M', 'B', 'B', 'B']
Length: 5, dtype: str

In [13]:
labels = np.array(train_labels[:5])

In [20]:
labels.shape[0]

5

In [ ]:

encoded_labels = np.zeros((labels.shape[0], 2), dtype=int)

In [15]:
encoded_labels

array([[0, 0],
       [0, 0],
       [0, 0],
       [0, 0],
       [0, 0]])

In [22]:
encoded_labels[labels == 'M', 0] = 1
encoded_labels[labels == 'B', 1] = 1

In [28]:
encoded_labels.tolist()

[[0, 1], [1, 0], [0, 1], [0, 1], [0, 1]]

In [24]:
labels

array(['B', 'M', 'B', 'B', 'B'], dtype=object)

In [21]:
def encode_labels(y) -> np.ndarray:
    """
    Converts categorical string labels into one-hot encoded binary targets.

    Why: A softmax output layer with categorical cross-entropy expects targets
    shaped like the network outputs. For this binary task, that means two columns
    instead of a single scalar label.

    Args:
        y (np.ndarray or list): Array of nominal labels (e.g., 'M' and 'B').

    Returns:
        np.ndarray: Array of shape (n_samples, 2) where 'M' -> [1, 0] and
        'B' -> [0, 1].
    """
    # Convert input to a NumPy array for consistent processing
    labels = np.array(y)
    # Initialize a zero matrix with two columns for one-hot encoding 
    # Shape[0] is the number of samples, and 2 is for the two classes (M and B)
    encoded = np.zeros((labels.shape[0], 2), dtype=int)
    encoded[labels == 'M', 0] = 1
    encoded[labels == 'B', 1] = 1
    return encoded


In [2]:

# Verification / Test
if __name__ == "__main__":
    dummy_labels = np.array(['B', 'M', 'B', 'B', 'M'])
    encoded = encode_labels(dummy_labels)
    
    print(f"Original: {dummy_labels}")
    print(f"Encoded:  {encoded}")
    assert np.all(encoded == np.array([0, 1, 0, 0, 1])), "Encoding failed!"
    print("Test passed: Labels accurately encoded to binary format.")

Original: ['B' 'M' 'B' 'B' 'M']
Encoded:  [0 1 0 0 1]
Test passed: Labels accurately encoded to binary format.


## Step 3: Scaler State & Fit (`fit`)

Next, we address standardizing numerical features via **Z-score Standardization**.

The formula for standardizing a feature $x$ is:
$$z = \frac{x - \mu}{\sigma}$$
Where:
- $\mu$ is the mean of the feature: $\mu = \frac{1}{N}\sum x_i$
- $\sigma$ is the standard deviation: $\sigma = \sqrt{\frac{1}{N}\sum (x_i - \mu)^2}$

**Crucial "Defense" Concept: Data Leakage**
We must calculate $\mu$ and $\sigma$ exclusively on the **training dataset** and store them statefully. If we calculated the mean/std over the entire dataset (train + validation), information about the validation set would "leak" into the training phase, giving the model an unfair advantage and artificially inflating our evaluation metrics.

Let's begin scaffolding the stateful `StandardScaler` class with its `fit` method.

In [3]:
class StandardScaler:
    """
    Stateful Scaler that standardizes features by removing the mean and scaling to unit variance.
    Calculates parameters (mean and std) exclusively on training data to prevent data leakage.
    """
    def __init__(self):
        # We initialize state variables to None.
        # They will be populated as 1D numpy arrays of shape (n_features,) after calling fit()
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        """
        Calculates and stores the mean and standard deviation for each feature.
        
        Why: We calculate this only once on the training data so that when we 
        evaluate performance on the validation set, the validation set is 
        transformed using the exact same metrics the model learned from.
        
        Args:
            X (np.ndarray): Training inputs of shape (n_samples, n_features)
        """
        # Ensure input is a numpy array
        X = np.array(X, dtype=float)
        
        # Calculate mean across rows (axis=0) to get average for each feature (column)
        # Math: mu = 1/N * sum(x_i)
        self.mean_ = np.mean(X, axis=0)
        
        # Calculate standard deviation across rows (axis=0)
        # Math: sigma = sqrt( 1/N * sum((x_i - mu)^2) )
        self.std_ = np.std(X, axis=0)


In [4]:

# Verification / Test
if __name__ == "__main__":
    # Create Dummy Data: 4 samples, 3 features
    # Feature 0: Values around 10
    # Feature 1: Values around 100
    # Feature 2: Values around 0.5
    dummy_X = np.array([
        [10.0, 100.0, 0.5],
        [12.0,  90.0, 0.6],
        [ 8.0, 110.0, 0.4],
        [10.0, 100.0, 0.5]
    ])
    
    scaler = StandardScaler()
    scaler.fit(dummy_X)
    
    print("Fitted Means:", scaler.mean_)
    print("Fitted Stds:", scaler.std_)


Fitted Means: [ 10.  100.    0.5]
Fitted Stds: [1.41421356 7.07106781 0.07071068]


In [6]:
    
assert scaler.mean_.shape == (3,), "Mean shape is incorrect."
assert scaler.std_.shape == (3,), "Standard deviation shape is incorrect."
print("Test passed: Scaler successfully computed stateful 'mean' and 'std' for each feature.")

Test passed: Scaler successfully computed stateful 'mean' and 'std' for each feature.


## Step 4: Transform Logic (`transform`)

Now we implement the `transform(self, X)` method to apply the Z-score equation:
$$z = \frac{x - \mu}{\sigma + \epsilon}$$

**Why use an Epsilon ($\epsilon$)?**
If a feature is completely constant across all training samples, its standard deviation ($\sigma$) will be exactly $0$. Dividing by zero causes `NaN` (Not a Number) errors that will instantly destroy the entire neural network (exploding gradients / weights turning to NaN). Adding a tiny number like $1 \times 10^{-8}$ prevents this division-by-zero without meaningfully altering the math for valid features.

**Why standardize at all?**
If one feature ranges from 0 to 1 and another ranges from 1,000 to 100,000, features with larger magnitudes will disproportionately dominate the gradient calculations during backpropagation. This causes the gradient descent algorithm to oscillate wildly or "vanish," preventing the model from converging. By squishing all features to a mean of $0$ and a standard deviation of $1$, we create a perfectly smooth "bowl" for gradient descent to navigate.

In [7]:
class StandardScaler:
    """
    Stateful Scaler that standardizes features by removing the mean and scaling to unit variance.
    Calculates parameters (mean and std) exclusively on training data to prevent data leakage.
    """
    def __init__(self):
        # Initialize state variables
        self.mean_ = None
        self.std_ = None
        # Tiny constant to prevent division by zero explicitly requested in constraints
        self.epsilon = 1e-8 

    def fit(self, X):
        """
        Calculates and stores the mean and standard deviation for each feature.
        """
        X = np.array(X, dtype=float)
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)

    def transform(self, X):
        """
        Applies standard scaling to the dataset using the previously fitted mean and std.
        
        Why: Standardizes the input magnitudes so gradient descent converges smoothly.
        Avoids dividing by zero using epsilon.
        
        Args:
            X (np.ndarray): Data to transform, shape (n_samples, n_features)
            
        Returns:
            np.ndarray: Scaled data, shape (n_samples, n_features)
        """
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Scaler has not been fitted yet. Call fit() before transform().")
            
        X = np.array(X, dtype=float)
        
        # Math: z = (x - mu) / (sigma + epsilon)
        # Vectorized operation applies the formula to the whole matrix instantly
        X_scaled = (X - self.mean_) / (self.std_ + self.epsilon)
        
        return X_scaled


In [8]:

# Verification / Test
if __name__ == "__main__":
    # Using the same dummy data
    dummy_X = np.array([
        [10.0, 100.0, 0.5],
        [12.0,  90.0, 0.6],
        [ 8.0, 110.0, 0.4],
        [10.0, 100.0, 0.5]
    ])
    
    scaler = StandardScaler()
    scaler.fit(dummy_X)
    X_scaled = scaler.transform(dummy_X)
    
    print("Original Data:\n", dummy_X)
    print("\nScaled Data:\n", X_scaled)
    
    # Mathematical assertion: 
    # Mean of scaled data should be very close to 0
    # Std of scaled data should be very close to 1
    assert np.allclose(np.mean(X_scaled, axis=0), 0), "Mean is not 0!"
    assert np.allclose(np.std(X_scaled, axis=0), 1), "Std is not 1!"
    print("\nTest passed: Scaled data has mean ~ 0 and std ~ 1.")

Original Data:
 [[ 10.  100.    0.5]
 [ 12.   90.    0.6]
 [  8.  110.    0.4]
 [ 10.  100.    0.5]]

Scaled Data:
 [[ 0.          0.          0.        ]
 [ 1.41421355 -1.41421356  1.41421336]
 [-1.41421355  1.41421356 -1.41421336]
 [ 0.          0.          0.        ]]

Test passed: Scaled data has mean ~ 0 and std ~ 1.


## Step 5: State Persistence (`save` / `load`)

In a real-world scenario, you train your model once and deploy it to make predictions later. The separate prediction program needs to apply the exact same transformation to incoming patient data as was applied to the training data. If we fitted a new scaler on a single patient's data, $\mu$ would equal the patient's data and $\sigma$ would be $0$, breaking the network completely.

Therefore, we must persist the fitted mean and standard deviation matrices to the disk. We will serialize our numpy arrays to a JSON file.

*Technical Detail*: NumPy arrays cannot be directly written to JSON. We must convert them to standard Python lists using `.tolist()` when saving, and convert them back to `np.array()` when loading.

In [9]:
import json
import os

class StandardScaler:
    """
    Stateful Scaler that standardizes features by removing the mean and scaling to unit variance.
    """
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self.epsilon = 1e-8
    # ──────────────────────────────────────────────────────────────────────
    def fit(self, X):
        """
        Calculates and stores the mean and standard deviation for each feature (axis=0).
        This will give us two 1D arrays of shape (n_features,) that we can use to standardize any dataset
        one value per feature.
        
        Why: These statistics are learned exclusively from training data to prevent
        data leakage. During transform(), we'll use these exact values to standardize
        both training and validation data identically, ensuring the network receives
        consistently scaled inputs across all phases.
        
        Args:
            X (np.ndarray): Training data, shape (n_samples, n_features)
        """
        X = np.array(X, dtype=float)
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)

    # ──────────────────────────────────────────────────────────────────────
    def transform(self, X):
        """
        Applies standard scaling (standardization) to the dataset using the previously fitted mean and std.
        
        Why: Standardizes input magnitudes so gradient descent converges smoothly during 
        backpropagation. Prevents features with larger magnitudes from dominating gradient 
        calculations. Uses epsilon to avoid division-by-zero errors.
        
        Args:
            X (np.ndarray): Data to transform, shape (n_samples, n_features)
            
        Returns:
            np.ndarray: Scaled data, shape (n_samples, n_features)
        """
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Scaler has not been fitted yet.")
        X = np.array(X, dtype=float)
        return (X - self.mean_) / (self.std_ + self.epsilon)

    # ──────────────────────────────────────────────────────────────────────────────

    def save(self, filepath):
        """
        Serializes the learned mean and standard deviation to a JSON file.
        Why: We need to preserve these exactly for the separate prediction script.
        """
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Scaler is not fitted, nothing to save.")
            
        data = {
            # Convert NumPy arrays to lists for JSON serialization
            'mean': self.mean_.tolist(),
            'std': self.std_.tolist()
        }
        
        # Ensure the directory exists
        os.makedirs(os.path.dirname(os.path.abspath(filepath)) or '.', exist_ok=True)
        
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=4)

    def load(self, filepath):
        """
        Deserializes the mean and standard deviation from a JSON file.
        Why: Allows a new instance of the scaler to instantly prepare validation data.
        """
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        # Convert lists back to NumPy arrays
        self.mean_ = np.array(data['mean'], dtype=float)
        self.std_ = np.array(data['std'], dtype=float)


# Verification / Test
if __name__ == "__main__":
    test_filepath = "test_scaler.json"
    
    # 1. Create and fit original scaler
    dummy_X = np.array([[10.0, 100.0, 0.5], [12.0, 90.0, 0.6], [8.0, 110.0, 0.4], [10.0, 100.0, 0.5]])
    scaler1 = StandardScaler()
    scaler1.fit(dummy_X)
    scaler1.save(test_filepath)
    print(f"Scaler 1 saved to {test_filepath}")
    
    # 2. Create new scaler and load states
    scaler2 = StandardScaler()
    scaler2.load(test_filepath)
    print("\nScaler 2 loaded from file.")
    
    # 3. Verify exactly matching states
    assert np.allclose(scaler1.mean_, scaler2.mean_), "Loaded mean does not match!"
    assert np.allclose(scaler1.std_, scaler2.std_), "Loaded std does not match!"
    print("Test passed: State perfectly persevered across serialization!")
    
    # Cleanup
    if os.path.exists(test_filepath):
        os.remove(test_filepath)

Scaler 1 saved to test_scaler.json

Scaler 2 loaded from file.
Test passed: State perfectly persevered across serialization!


## Step 6 & 7: Final Verification and Extraction

We have successfully implemented and validated the mathematics behind Label Encoding and Z-score Standardization using pure `numpy` and `json`. 

The test blocks in the previous Python cells confirm:
1. `encode_labels` correctly maps `'M'` -> 1 and `'B'` -> 0.
2. `StandardScaler` standardizes dummy features to $\mu \approx 0$ and $\sigma \approx 1$.
3. The calculated parameters can be accurately saved to and loaded from a JSON file.

We will now extract these finalized, documented components into a standalone module: `src/scaler.py`.